In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build top pairs summary (includes counts and per-pair first/last timestamps)
def top_pairs_summary(df, id_col, name_col, top_n=20):
    grp = df.groupby([id_col, name_col], dropna=False)
    counts = grp.size().reset_index(name='count')
    ts_bounds = grp['timestamp'].agg(first_ts='min', last_ts='max').reset_index()
    merged = counts.merge(ts_bounds, on=[id_col, name_col])
    merged_sorted = merged.sort_values('count', ascending=False).head(top_n).copy()
    # format timestamps as ISO strings for compact JSON-like output
    merged_sorted['first_ts'] = merged_sorted['first_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    merged_sorted['last_ts'] = merged_sorted['last_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Rename columns for clarity
    merged_sorted = merged_sorted.rename(columns={id_col: 'cmdb_id', name_col: 'name'})
    # Keep only relevant columns
    return merged_sorted[['cmdb_id', 'name', 'count', 'first_ts', 'last_ts']]

# Build summaries
metric_rows = metric_df.shape[0]
metric_min_ts = metric_df['timestamp'].min()
metric_max_ts = metric_df['timestamp'].max()
metric_top = top_pairs_summary(metric_df, 'cmdb_id', 'kpi_name', top_n=20)

trace_rows = trace_df.shape[0]
trace_min_ts = trace_df['timestamp'].min()
trace_max_ts = trace_df['timestamp'].max()
trace_top = top_pairs_summary(trace_df, 'cmdb_id', 'trace_name', top_n=20)

log_rows = log_df.shape[0]
log_min_ts = log_df['timestamp'].min()
log_max_ts = log_df['timestamp'].max()
log_top = top_pairs_summary(log_df, 'cmdb_id', 'log_name', top_n=20)

error_rows = error_df.shape[0]
error_min_ts = error_df['timestamp'].min()
error_max_ts = error_df['timestamp'].max()
error_top = top_pairs_summary(error_df, 'cmdb_id', 'message', top_n=20)

# Construct concise dict-like summary
summary = {
    'metric.csv': {
        'rows': int(metric_rows),
        'timestamp_min': metric_min_ts.isoformat(),
        'timestamp_max': metric_max_ts.isoformat(),
        'top_pairs_count': metric_top.to_dict(orient='records')  # up to 20 entries
    },
    'trace.csv': {
        'rows': int(trace_rows),
        'timestamp_min': trace_min_ts.isoformat(),
        'timestamp_max': trace_max_ts.isoformat(),
        'top_pairs_count': trace_top.to_dict(orient='records')
    },
    'log.csv': {
        'rows': int(log_rows),
        'timestamp_min': log_min_ts.isoformat(),
        'timestamp_max': log_max_ts.isoformat(),
        'top_pairs_count': log_top.to_dict(orient='records')
    },
    'error_logs.csv': {
        'rows': int(error_rows),
        'timestamp_min': error_min_ts.isoformat(),
        'timestamp_max': error_max_ts.isoformat(),
        'top_pairs_count': error_top.to_dict(orient='records')
    }
}

summary

```
Out[1]:
```
Summary (plain English)

- metric.csv
  - Rows: 1,900
  - Timestamp range (UTC): 2024-01-19T19:15:00+00:00 → 2024-01-19T19:39:00+00:00
  - Top observed (cmdb_id, kpi_name) patterns (most have count = 25 and cover the full time range): examples include
    - adservice: cpu, diskio, error, latency-50, latency-90, mem, socket, workload
    - cartservice: cpu, diskio, latency-50, latency-90, mem, socket, workload
    - checkoutservice: cpu, latency-50, latency-90, mem, socket
  - Overall: many services report the same set of per-minute KPIs (most pairs appear 25 times across the 25-minute window).

- trace.csv
  - Rows: 3,800
  - Timestamp range (UTC): 2024-01-19T19:15:00+00:00 → 2024-01-19T19:39:00+00:00
  - Top observed (cmdb_id, trace_name) patterns (top entries all count = 25): primarily from checkoutservice, e.g.
    - trace.from_checkoutservice.* (duration_mean, duration_p95, error_rate, row_count)
    - trace.from_frontendservice.*, trace.from_root.*, trace.to_checkoutservice.*, trace.to_currencyservice.* 
  - Overall: trace telemetry is concentrated in checkoutservice with consistent per-minute trace metrics over the time window.

- log.csv
  - Rows: 460
  - Timestamp range (UTC): 2024-01-19T19:15:00+00:00 → 2024-01-19T19:39:00+00:00
  - Top observed (cmdb_id, log_name) patterns: most services report log.error_count and log.row_count with count = 25 (full window), examples:
    - adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice (log.error_count and log.row_count)
    - redis has fewer entries: log.row_count and log.error_count with count = 5 (first seen 2024-01-19T19:18:00Z).
  - Overall: broad per-minute logging across services; redis data starts later and is sparser.

- error_logs.csv
  - Rows: 0
  - Timestamp range: none (no entries)
  - Top pairs: none

If you want, I can next: (a) show the full top-20 lists for any file, or (b) extract a per-service compact table of which KPIs/traces/logs exist.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'rows': 1900, 'timestamp_min': '2024-01-19T19:15:00+00:00', 'timestamp_max': '2024-01-19T19:39:00+00:00', 'top_pairs_count': [{'cmdb_id': 'adservice', 'name': 'cpu', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'diskio', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'error', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'latency-50', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'latency-90', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'mem', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'socket', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'workload', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'cpu', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'diskio', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'latency-50', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'latency-90', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'mem', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'socket', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'workload', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'cpu', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'latency-50', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'latency-90', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'mem', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'socket', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}]}, 'trace.csv': {'rows': 3800, 'timestamp_min': '2024-01-19T19:15:00+00:00', 'timestamp_max': '2024-01-19T19:39:00+00:00', 'top_pairs_count': [{'cmdb_id': 'checkoutservice', 'name': 'trace.from_checkoutservice.duration_mean', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_checkoutservice.duration_p95', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_checkoutservice.error_rate', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_checkoutservice.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_frontendservice.duration_mean', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_frontendservice.duration_p95', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_frontendservice.error_rate', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_frontendservice.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_root.duration_mean', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_root.duration_p95', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_root.error_rate', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.from_root.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_checkoutservice.duration_mean', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_checkoutservice.duration_p95', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_checkoutservice.error_rate', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_checkoutservice.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_currencyservice.duration_mean', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_currencyservice.duration_p95', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_currencyservice.error_rate', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'trace.to_currencyservice.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}]}, 'log.csv': {'rows': 460, 'timestamp_min': '2024-01-19T19:15:00+00:00', 'timestamp_max': '2024-01-19T19:39:00+00:00', 'top_pairs_count': [{'cmdb_id': 'adservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'adservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'cartservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'checkoutservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'currencyservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'currencyservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'emailservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'emailservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'frontend', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'frontend', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'paymentservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'paymentservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'recommendationservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'recommendationservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'shippingservice', 'name': 'log.row_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'shippingservice', 'name': 'log.error_count', 'count': 25, 'first_ts': '2024-01-19T19:15:00Z', 'last_ts': '2024-01-19T19:39:00Z'}, {'cmdb_id': 'redis', 'name': 'log.row_count', 'count': 5, 'first_ts': '2024-01-19T19:18:00Z', 'last_ts': '2024-01-19T19:38:00Z'}, {'cmdb_id': 'redis', 'name': 'log.error_count', 'count': 5, 'first_ts': '2024-01-19T19:18:00Z', 'last_ts': '2024-01-19T19:38:00Z'}]}, 'error_logs.csv': {'rows': 0, 'timestamp_min': 'NaT', 'timestamp_max': 'NaT', 'top_pairs_count': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df if available; otherwise load it
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp parsed to UTC datetimes
if not pd.api.types.is_datetime64_any_dtype(metric_df['timestamp']):
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    # ensure timezone-aware UTC
    metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert('UTC')

# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Filter metric rows for candidates (for grouping/pairs discovery)
metric_candidates_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 1) Compute full-series count and P95 threshold using all available rows for each (cmdb_id, kpi_name)
full_counts = metric_candidates_df.groupby(['cmdb_id', 'kpi_name'], dropna=False).size().reset_index(name='full_count')
p95 = metric_candidates_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95_threshold')

full_series_stats_df = full_counts.merge(p95, on=['cmdb_id', 'kpi_name'])

# 2) Convert timestamps (already done) and filter to incident window
start_ts = pd.to_datetime('2024-01-19T19:12:14Z', utc=True)
end_ts = pd.to_datetime('2024-01-19T19:42:14Z', utc=True)
incident_metric_df = metric_candidates_df[(metric_candidates_df['timestamp'] >= start_ts) & (metric_candidates_df['timestamp'] <= end_ts)].copy()

# 3) For each pair compute incident_count, anomaly_count, earliest_anomaly_ts, max_value_in_incident
# Aggregate incident metrics
incident_agg = incident_metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    incident_count = ('value', 'size'),
    max_value_in_incident = ('value', 'max')
).reset_index()

# Determine anomaly_count and earliest anomaly timestamp per pair
# Merge p95 thresholds first to evaluate anomalies
incident_with_p95 = incident_metric_df.merge(p95, on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomalies (value > p95_threshold). If p95 is NaN (no full series), treat as no anomaly.
incident_with_p95['is_anomaly'] = False
mask_valid_p95 = incident_with_p95['p95_threshold'].notna()
incident_with_p95.loc[mask_valid_p95, 'is_anomaly'] = incident_with_p95.loc[mask_valid_p95, 'value'] > incident_with_p95.loc[mask_valid_p95, 'p95_threshold']

anomaly_counts = incident_with_p95[incident_with_p95['is_anomaly']].groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    anomaly_count = ('is_anomaly', 'sum'),
    earliest_anomaly_ts = ('timestamp', 'min')
).reset_index()

# Merge all aggregates
pair_incident_agg_df = full_series_stats_df.merge(incident_agg, on=['cmdb_id', 'kpi_name'], how='left')
pair_incident_agg_df = pair_incident_agg_df.merge(anomaly_counts, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for pairs with no incident rows
pair_incident_agg_df['incident_count'] = pair_incident_agg_df['incident_count'].fillna(0).astype(int)
pair_incident_agg_df['max_value_in_incident'] = pair_incident_agg_df['max_value_in_incident']  # may be NaN
pair_incident_agg_df['anomaly_count'] = pair_incident_agg_df['anomaly_count'].fillna(0).astype(int)
pair_incident_agg_df['earliest_anomaly_ts'] = pair_incident_agg_df['earliest_anomaly_ts']  # may be NaT

# Compute ratio_max_to_p95 (null if p95 == 0 or p95 is NaN or max is NaN)
def compute_ratio(row):
    p95v = row['p95_threshold']
    maxv = row['max_value_in_incident']
    if pd.isna(p95v) or p95v == 0 or pd.isna(maxv):
        return None
    return float(maxv) / float(p95v)

pair_incident_agg_df['ratio_max_to_p95'] = pair_incident_agg_df.apply(compute_ratio, axis=1)

# Format earliest_anomaly_ts as ISO string or None
pair_incident_agg_df['earliest_anomaly_ts_iso'] = pair_incident_agg_df['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
pair_incident_agg_df.loc[pair_incident_agg_df['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

# Keep only records with incident_count > 0
results_df = pair_incident_agg_df[pair_incident_agg_df['incident_count'] > 0].copy()

# Select and order columns as requested
results_df = results_df[[
    'cmdb_id', 'kpi_name', 'full_count', 'p95_threshold',
    'incident_count', 'anomaly_count', 'earliest_anomaly_ts_iso',
    'max_value_in_incident', 'ratio_max_to_p95'
]]

# Sort by anomaly_count desc, then ratio desc (NaNs go last)
results_df = results_df.sort_values(by=['anomaly_count', 'ratio_max_to_p95'], ascending=[False, False])

# Limit to top 20 rows
results_df = results_df.head(20).reset_index(drop=True)

# Prepare JSON-like list for return (compact)
results_list = results_df.rename(columns={'earliest_anomaly_ts_iso':'earliest_anomaly_ts'}).to_dict(orient='records')

# Keep intermediate DataFrames in kernel variables for follow-up:
# - full_series_stats_df (full-series counts and p95)
# - incident_metric_df (filtered incident-window raw rows for candidates)
# - incident_with_p95 (per-incident-row merged with p95 and anomaly flag)
# - pair_incident_agg_df (merged aggregates before final filtering)
# - results_df (final compact DataFrame)
full_series_stats_df, incident_metric_df, incident_with_p95, pair_incident_agg_df, results_df, results_list

```
Out[2]:
```
Summary (plain English)

- I analyzed metrics for the candidate services (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice) within the incident window (2024-01-19T19:12:14Z → 2024-01-19T19:42:14Z) and computed per-(cmdb_id, kpi_name) p95 thresholds from the full available series (no pre-filtering).

- Overall results: 20 (cmdb_id, kpi_name) records had incident rows and are included. For every reported pair full_count = 25 and incident_count = 25 (the full series covers the incident window).

- Anomalies: every returned pair had anomaly_count = 2 (i.e., two samples in the incident window exceeded the full-series p95). There are no pairs with >2 anomalies.

- Severity (ratio of max_in_window to p95): the exceedances are small-to-moderate. The largest ratio is currencyservice / latency-90 ≈ 1.182 (max 0.13699 vs p95 0.11590). Most other ratios are close to 1.01–1.07 (small percent increases over p95).

- Earliest anomaly times (examples):
  - currencyservice / latency-90: first anomaly at 2024-01-19T19:28:00Z (ratio ≈ 1.182).
  - checkoutservice / workload: first anomaly at 2024-01-19T19:18:00Z (ratio ≈ 1.066).
  - checkoutservice / latency-50: first anomaly at 2024-01-19T19:28:00Z (ratio ≈ 1.064).
  - Other services (productcatalogservice, recommendationservice, emailservice) show earliest anomalies mostly between 19:15 and 19:33 UTC (ratios ~1.01–1.04).

- Notable KPI types impacted: latency metrics (latency-50, latency-90), workload, cpu, mem, diskio, socket across the candidate services — but the anomalies are mild (small exceedances of p95).

- Conclusion: Multiple KPIs across the candidate services show two brief exceedances above their historical p95, with the largest relative deviation in currencyservice latency-90 (~18% above p95). No single KPI shows large or sustained spikes in this analysis. If you want, I can present the compact JSON-like table of the 20 records again or expand timestamps/values for any specific service/KPI.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  full_count  p95_threshold
0         checkoutservice         cpu          25   4.757836e-01
1         checkoutservice  latency-50          25   2.655181e-01
2         checkoutservice  latency-90          25   7.540155e-01
3         checkoutservice         mem          25   1.205532e+07
4         checkoutservice      socket          25   9.000000e+00
..                    ...         ...         ...            ...
29  recommendationservice  latency-50          25   7.652374e-03
30  recommendationservice  latency-90          25   9.847707e-03
31  recommendationservice         mem          25   4.593720e+07
32  recommendationservice      socket          25   4.000000e+00
33  recommendationservice    workload          25   1.975139e+01

[34 rows x 4 columns],                      timestamp                cmdb_id    kpi_name         value
15   2024-01-19 19:15:00+00:00        checkoutservice         cpu  4.263772e-01
16   2024-01-19 19:15:00+00:00        checkoutservice  latency-50  1.004245e-01
17   2024-01-19 19:15:00+00:00        checkoutservice  latency-90  2.196986e-01
18   2024-01-19 19:15:00+00:00        checkoutservice         mem  1.158091e+07
19   2024-01-19 19:15:00+00:00        checkoutservice      socket  9.000000e+00
...                        ...                    ...         ...           ...
1885 2024-01-19 19:39:00+00:00  recommendationservice  latency-50  7.601963e-03
1886 2024-01-19 19:39:00+00:00  recommendationservice  latency-90  9.771458e-03
1887 2024-01-19 19:39:00+00:00  recommendationservice         mem  4.592927e+07
1888 2024-01-19 19:39:00+00:00  recommendationservice      socket  4.000000e+00
1889 2024-01-19 19:39:00+00:00  recommendationservice    workload  1.780920e+01

[850 rows x 4 columns],                     timestamp                cmdb_id    kpi_name         value  p95_threshold  is_anomaly
0   2024-01-19 19:15:00+00:00        checkoutservice         cpu  4.263772e-01   4.757836e-01       False
1   2024-01-19 19:15:00+00:00        checkoutservice  latency-50  1.004245e-01   2.655181e-01       False
2   2024-01-19 19:15:00+00:00        checkoutservice  latency-90  2.196986e-01   7.540155e-01       False
3   2024-01-19 19:15:00+00:00        checkoutservice         mem  1.158091e+07   1.205532e+07       False
4   2024-01-19 19:15:00+00:00        checkoutservice      socket  9.000000e+00   9.000000e+00       False
..                        ...                    ...         ...           ...            ...         ...
845 2024-01-19 19:39:00+00:00  recommendationservice  latency-50  7.601963e-03   7.652374e-03       False
846 2024-01-19 19:39:00+00:00  recommendationservice  latency-90  9.771458e-03   9.847707e-03       False
847 2024-01-19 19:39:00+00:00  recommendationservice         mem  4.592927e+07   4.593720e+07       False
848 2024-01-19 19:39:00+00:00  recommendationservice      socket  4.000000e+00   4.000000e+00       False
849 2024-01-19 19:39:00+00:00  recommendationservice    workload  1.780920e+01   1.975139e+01       False

[850 rows x 6 columns],                   cmdb_id    kpi_name  full_count  p95_threshold  incident_count  max_value_in_incident  anomaly_count       earliest_anomaly_ts  ratio_max_to_p95 earliest_anomaly_ts_iso
0         checkoutservice         cpu          25   4.757836e-01              25           4.776510e-01              2 2024-01-19 19:18:00+00:00          1.003925    2024-01-19T19:18:00Z
1         checkoutservice  latency-50          25   2.655181e-01              25           2.825671e-01              2 2024-01-19 19:28:00+00:00          1.064211    2024-01-19T19:28:00Z
2         checkoutservice  latency-90          25   7.540155e-01              25           7.973810e-01              2 2024-01-19 19:36:00+00:00          1.057513    2024-01-19T19:36:00Z
3         checkoutservice         mem          25   1.205532e+07              25           1.225926e+07              2 2024-01-19 19:17:00+00:00          1.016917    2024-01-19T19:17:00Z
4         checkoutservice      socket          25   9.000000e+00              25           9.000000e+00              0                       NaT          1.000000                    None
..                    ...         ...         ...            ...             ...                    ...            ...                       ...               ...                     ...
29  recommendationservice  latency-50          25   7.652374e-03              25           7.700449e-03              2 2024-01-19 19:32:00+00:00          1.006282    2024-01-19T19:32:00Z
30  recommendationservice  latency-90          25   9.847707e-03              25           1.013170e-02              2 2024-01-19 19:32:00+00:00          1.028838    2024-01-19T19:32:00Z
31  recommendationservice         mem          25   4.593720e+07              25           4.598231e+07              2 2024-01-19 19:27:00+00:00          1.000982    2024-01-19T19:27:00Z
32  recommendationservice      socket          25   4.000000e+00              25           4.000000e+00              0                       NaT          1.000000                    None
33  recommendationservice    workload          25   1.975139e+01              25           2.031668e+01              2 2024-01-19 19:15:00+00:00          1.028621    2024-01-19T19:15:00Z

[34 rows x 10 columns],                   cmdb_id    kpi_name  full_count  p95_threshold  incident_count  anomaly_count earliest_anomaly_ts_iso  max_value_in_incident  ratio_max_to_p95
0         currencyservice  latency-90          25   1.159013e-01              25              2    2024-01-19T19:28:00Z           1.369899e-01          1.181953
1         checkoutservice    workload          25   1.138593e+00              25              2    2024-01-19T19:18:00Z           1.213333e+00          1.065642
2         checkoutservice  latency-50          25   2.655181e-01              25              2    2024-01-19T19:28:00Z           2.825671e-01          1.064211
3         checkoutservice  latency-90          25   7.540155e-01              25              2    2024-01-19T19:36:00Z           7.973810e-01          1.057513
4   productcatalogservice    workload          25   1.489133e+02              25              2    2024-01-19T19:15:00Z           1.544824e+02          1.037398
5   recommendationservice  latency-90          25   9.847707e-03              25              2    2024-01-19T19:32:00Z           1.013170e-02          1.028838
6   recommendationservice    workload          25   1.975139e+01              25              2    2024-01-19T19:15:00Z           2.031668e+01          1.028621
7         currencyservice      diskio          25   1.181916e+10              25              2    2024-01-19T19:34:00Z           1.214236e+10          1.027345
8            emailservice    workload          25   1.170243e+00              25              2    2024-01-19T19:18:00Z           1.201000e+00          1.026282
9         currencyservice      socket          25   7.133333e+00              25              2    2024-01-19T19:33:00Z           7.300000e+00          1.023364
10           emailservice         cpu          25   3.479524e-01              25              2    2024-01-19T19:16:00Z           3.547176e-01          1.019443
11        checkoutservice         mem          25   1.205532e+07              25              2    2024-01-19T19:17:00Z           1.225926e+07          1.016917
12  recommendationservice         cpu          25   3.038168e+00              25              2    2024-01-19T19:18:00Z           3.076082e+00          1.012479
13        currencyservice  latency-50          25   6.210293e-02              25              2    2024-01-19T19:28:00Z           6.267443e-02          1.009202
14  productcatalogservice  latency-50          25   1.975013e-03              25              2    2024-01-19T19:33:00Z           1.992816e-03          1.009014
15  productcatalogservice  latency-90          25   4.427304e-03              25              2    2024-01-19T19:33:00Z           4.466952e-03          1.008955
16  productcatalogservice         cpu          25   3.705899e+00              25              2    2024-01-19T19:15:00Z           3.738387e+00          1.008767
17           emailservice  latency-90          25   4.727724e-03              25              2    2024-01-19T19:19:00Z           4.762977e-03          1.007457
18           emailservice  latency-50          25   3.070958e-03              25              2    2024-01-19T19:19:00Z           3.090543e-03          1.006377
19  recommendationservice  latency-50          25   7.652374e-03              25              2    2024-01-19T19:32:00Z           7.700449e-03          1.006282, [{'cmdb_id': 'currencyservice', 'kpi_name': 'latency-90', 'full_count': 25, 'p95_threshold': 0.11590134726295388, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:28:00Z', 'max_value_in_incident': 0.1369899007149246, 'ratio_max_to_p95': 1.1819526170314962}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'workload', 'full_count': 25, 'p95_threshold': 1.1385933333333333, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:18:00Z', 'max_value_in_incident': 1.2133333333333334, 'ratio_max_to_p95': 1.065642400857198}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'latency-50', 'full_count': 25, 'p95_threshold': 0.2655180614579927, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:28:00Z', 'max_value_in_incident': 0.2825671203796204, 'ratio_max_to_p95': 1.0642105430719446}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'latency-90', 'full_count': 25, 'p95_threshold': 0.7540154907796452, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:36:00Z', 'max_value_in_incident': 0.7973809523809524, 'ratio_max_to_p95': 1.057512693215981}, {'cmdb_id': 'productcatalogservice', 'kpi_name': 'workload', 'full_count': 25, 'p95_threshold': 148.91334144927535, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:15:00Z', 'max_value_in_incident': 154.4824166666667, 'ratio_max_to_p95': 1.0373980945104797}, {'cmdb_id': 'recommendationservice', 'kpi_name': 'latency-90', 'full_count': 25, 'p95_threshold': 0.00984770743102134, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:32:00Z', 'max_value_in_incident': 0.0101316990328248, 'ratio_max_to_p95': 1.028838346771844}, {'cmdb_id': 'recommendationservice', 'kpi_name': 'workload', 'full_count': 25, 'p95_threshold': 19.751386376811592, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:15:00Z', 'max_value_in_incident': 20.316683333333334, 'ratio_max_to_p95': 1.0286206216483826}, {'cmdb_id': 'currencyservice', 'kpi_name': 'diskio', 'full_count': 25, 'p95_threshold': 11819161085.130486, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:34:00Z', 'max_value_in_incident': 12142360557.835562, 'ratio_max_to_p95': 1.0273453818233926}, {'cmdb_id': 'emailservice', 'kpi_name': 'workload', 'full_count': 25, 'p95_threshold': 1.1702433333333333, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:18:00Z', 'max_value_in_incident': 1.201, 'ratio_max_to_p95': 1.026282283171876}, {'cmdb_id': 'currencyservice', 'kpi_name': 'socket', 'full_count': 25, 'p95_threshold': 7.133333333333333, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:33:00Z', 'max_value_in_incident': 7.3, 'ratio_max_to_p95': 1.0233644859813085}, {'cmdb_id': 'emailservice', 'kpi_name': 'cpu', 'full_count': 25, 'p95_threshold': 0.34795239596502703, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:16:00Z', 'max_value_in_incident': 0.3547175731663007, 'ratio_max_to_p95': 1.0194428240176672}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'mem', 'full_count': 25, 'p95_threshold': 12055319.893333334, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:17:00Z', 'max_value_in_incident': 12259259.733333332, 'ratio_max_to_p95': 1.0169169994495773}, {'cmdb_id': 'recommendationservice', 'kpi_name': 'cpu', 'full_count': 25, 'p95_threshold': 3.03816813024478, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:18:00Z', 'max_value_in_incident': 3.0760823149801104, 'ratio_max_to_p95': 1.0124792911748026}, {'cmdb_id': 'currencyservice', 'kpi_name': 'latency-50', 'full_count': 25, 'p95_threshold': 0.06210293069257032, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:28:00Z', 'max_value_in_incident': 0.0626744289052313, 'ratio_max_to_p95': 1.009202435477483}, {'cmdb_id': 'productcatalogservice', 'kpi_name': 'latency-50', 'full_count': 25, 'p95_threshold': 0.00197501303011216, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:33:00Z', 'max_value_in_incident': 0.0019928160401058, 'ratio_max_to_p95': 1.0090141228043588}, {'cmdb_id': 'productcatalogservice', 'kpi_name': 'latency-90', 'full_count': 25, 'p95_threshold': 0.00442730410622358, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:33:00Z', 'max_value_in_incident': 0.004466951691756, 'ratio_max_to_p95': 1.0089552433221576}, {'cmdb_id': 'productcatalogservice', 'kpi_name': 'cpu', 'full_count': 25, 'p95_threshold': 3.7058992887898965, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:15:00Z', 'max_value_in_incident': 3.738387318000483, 'ratio_max_to_p95': 1.008766570993675}, {'cmdb_id': 'emailservice', 'kpi_name': 'latency-90', 'full_count': 25, 'p95_threshold': 0.00472772449055304, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:19:00Z', 'max_value_in_incident': 0.0047629768392741, 'ratio_max_to_p95': 1.0074565150299053}, {'cmdb_id': 'emailservice', 'kpi_name': 'latency-50', 'full_count': 25, 'p95_threshold': 0.0030709580503072197, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:19:00Z', 'max_value_in_incident': 0.0030905426884856, 'ratio_max_to_p95': 1.0063773707935937}, {'cmdb_id': 'recommendationservice', 'kpi_name': 'latency-50', 'full_count': 25, 'p95_threshold': 0.00765237367078674, 'incident_count': 25, 'anomaly_count': 2, 'earliest_anomaly_ts': '2024-01-19T19:32:00Z', 'max_value_in_incident': 0.0077004490058241, 'ratio_max_to_p95': 1.0062824081919692}])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse trace_df if available; otherwise load it
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamp parsed to UTC datetimes
if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    trace_df['timestamp'] = trace_df['timestamp'].dt.tz_convert('UTC')

# 1) Compute global full-series counts and P95 per (cmdb_id, trace_name) BEFORE any time filtering
trace_full_counts = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False).size().reset_index(name='full_count')
trace_p95 = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95_threshold')
trace_full_stats_df = trace_full_counts.merge(trace_p95, on=['cmdb_id', 'trace_name'])

# 2) Filter trace rows to the incident window
start_ts = pd.to_datetime('2024-01-19T19:12:14Z', utc=True)
end_ts = pd.to_datetime('2024-01-19T19:42:14Z', utc=True)
incident_trace_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) For each pair compute incident_count, anomaly_count, earliest anomaly ts, max value in incident
incident_agg = incident_trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    incident_count = ('value', 'size'),
    max_value_in_incident = ('value', 'max')
).reset_index()

# Merge p95 thresholds into incident rows to flag anomalies
incident_with_p95_trace = incident_trace_df.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# Flag anomalies: value > p95_threshold (only when p95 exists)
incident_with_p95_trace['is_anomaly'] = False
mask_valid_p95 = incident_with_p95_trace['p95_threshold'].notna()
incident_with_p95_trace.loc[mask_valid_p95, 'is_anomaly'] = incident_with_p95_trace.loc[mask_valid_p95, 'value'] > incident_with_p95_trace.loc[mask_valid_p95, 'p95_threshold']

anomaly_counts = incident_with_p95_trace[incident_with_p95_trace['is_anomaly']].groupby(
    ['cmdb_id', 'trace_name'], dropna=False
).agg(
    anomaly_count = ('is_anomaly', 'sum'),
    earliest_anomaly_ts = ('timestamp', 'min')
).reset_index()

# Merge all aggregates
pair_incident_agg_trace = trace_full_stats_df.merge(incident_agg, on=['cmdb_id', 'trace_name'], how='left')
pair_incident_agg_trace = pair_incident_agg_trace.merge(anomaly_counts, on=['cmdb_id', 'trace_name'], how='left')

# Fill NaNs for pairs with no incident rows
pair_incident_agg_trace['incident_count'] = pair_incident_agg_trace['incident_count'].fillna(0).astype(int)
pair_incident_agg_trace['max_value_in_incident'] = pair_incident_agg_trace['max_value_in_incident']  # may be NaN
pair_incident_agg_trace['anomaly_count'] = pair_incident_agg_trace['anomaly_count'].fillna(0).astype(int)
pair_incident_agg_trace['earliest_anomaly_ts'] = pair_incident_agg_trace['earliest_anomaly_ts']  # may be NaT

# Compute ratio_max_to_p95 (null if p95 == 0 or p95 is NaN or max is NaN)
def compute_ratio_trace(row):
    p95v = row['p95_threshold']
    maxv = row['max_value_in_incident']
    if pd.isna(p95v) or p95v == 0 or pd.isna(maxv):
        return None
    return float(maxv) / float(p95v)

pair_incident_agg_trace['ratio_max_to_p95'] = pair_incident_agg_trace.apply(compute_ratio_trace, axis=1)

# Format earliest anomaly ts as ISO string or None
pair_incident_agg_trace['earliest_anomaly_ts_iso'] = pair_incident_agg_trace['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
pair_incident_agg_trace.loc[pair_incident_agg_trace['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

# Filter to records with incident_count > 0
results_df_trace = pair_incident_agg_trace[pair_incident_agg_trace['incident_count'] > 0].copy()

# Select and order columns
results_df_trace = results_df_trace[[
    'cmdb_id', 'trace_name', 'full_count', 'p95_threshold',
    'incident_count', 'anomaly_count', 'earliest_anomaly_ts_iso',
    'max_value_in_incident', 'ratio_max_to_p95'
]]

# Sort by anomaly_count desc then ratio desc (NaNs last), limit to top 20
results_df_trace = results_df_trace.sort_values(by=['anomaly_count', 'ratio_max_to_p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# Rename earliest column for clarity in output
results_df_trace = results_df_trace.rename(columns={'earliest_anomaly_ts_iso': 'earliest_anomaly_ts'})

# Keep intermediate DataFrames in kernel for follow-up:
# trace_full_stats_df, incident_trace_df, incident_with_p95_trace, pair_incident_agg_trace, results_df_trace

results_df_trace

```
Out[3]:
```
Summary (plain English)

- Analysis scope: computed global P95 per (cmdb_id, trace_name) from the full trace series, then inspected trace rows in the incident window 2024-01-19T19:12:14Z → 2024-01-19T19:42:14Z.

- Overall pattern: every returned (cmdb_id, trace_name) has full_count = 25 and incident_count = 25 (the full series covers the window). Each returned pair shows anomaly_count = 2 (two samples in the incident window exceeded the global P95).

- Top anomalies (ordered by anomaly_count then ratio): strongest relative spikes are concentrated around 19:32:00Z and affect currency-related traces:
  1. currencyservice / trace.from_root.duration_p95 — p95 ≈ 0.000632, max_in_window ≈ 0.026385, earliest anomaly 2024-01-19T19:32:00Z, ratio ≈ 41.76 (very large relative spike).
  2. root / trace.to_currencyservice.duration_p95 — same stats as above (same span observed from the root side).
  3. currencyservice / trace.from_root.duration_mean — p95 ≈ 0.000164, max ≈ 0.004944, earliest anomaly 2024-01-19T19:32:00Z, ratio ≈ 30.22.
  4. root / trace.to_currencyservice.duration_mean — same as above from root side.
  5. checkoutservice / trace.to_currencyservice.duration_p95 — p95 ≈ 0.001668, max ≈ 0.022723, earliest anomaly 2024-01-19T19:28:00Z, ratio ≈ 13.62.
  6. currencyservice / trace.from_checkoutservice.duration_p95 — mirrors #5 (same spike seen from currencyservice side).

- Other notable spikes (moderate ratios 1.3–1.5) include checkoutservice traces (trace.from_checkoutservice.duration_mean and to_checkoutservice.duration_mean), some root/payment traces, and the frontend ↔ checkoutservice paths (earliest anomalies mostly between ~19:15 and ~19:36 UTC).

- Timing: the most extreme anomalies occur at ~19:32:00Z (currency-related traces) and ~19:28–19:36Z for checkout ↔ currency and frontend ↔ checkout traces. Each anomaly is brief (2 samples exceed P95).

- Conclusion: trace telemetry points to large, short-lived latency spikes involving currencyservice (and traces to/from it) around 19:32Z, with additional moderate spikes on checkoutservice and frontend traces in the 19:28–19:36Z timeframe. These currency-related trace spikes are the most severe by ratio.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  full_count  p95_threshold  incident_count  anomaly_count   earliest_anomaly_ts  max_value_in_incident  ratio_max_to_p95
0         currencyservice                    trace.from_root.duration_p95          25       0.000632              25              2  2024-01-19T19:32:00Z               0.026385         41.756002
1                    root           trace.to_currencyservice.duration_p95          25       0.000632              25              2  2024-01-19T19:32:00Z               0.026385         41.756002
2         currencyservice                   trace.from_root.duration_mean          25       0.000164              25              2  2024-01-19T19:32:00Z               0.004944         30.219030
3                    root          trace.to_currencyservice.duration_mean          25       0.000164              25              2  2024-01-19T19:32:00Z               0.004944         30.219030
4         checkoutservice           trace.to_currencyservice.duration_p95          25       0.001668              25              2  2024-01-19T19:28:00Z               0.022723         13.618699
5         currencyservice         trace.from_checkoutservice.duration_p95          25       0.001668              25              2  2024-01-19T19:28:00Z               0.022723         13.618699
6          paymentservice                    trace.from_root.duration_p95          25       0.000115              25              2  2024-01-19T19:15:00Z               0.000178          1.546750
7                    root            trace.to_paymentservice.duration_p95          25       0.000115              25              2  2024-01-19T19:15:00Z               0.000178          1.546750
8         checkoutservice        trace.from_checkoutservice.duration_mean          25       0.009383              25              2  2024-01-19T19:36:00Z               0.014210          1.514428
9         checkoutservice          trace.to_checkoutservice.duration_mean          25       0.009383              25              2  2024-01-19T19:36:00Z               0.014210          1.514428
10  productcatalogservice  trace.from_recommendationservice.duration_mean          25       0.000018              25              2  2024-01-19T19:34:00Z               0.000026          1.461993
11  recommendationservice    trace.to_productcatalogservice.duration_mean          25       0.000018              25              2  2024-01-19T19:34:00Z               0.000026          1.461993
12        checkoutservice         trace.from_frontendservice.duration_p95          25       0.774182              25              2  2024-01-19T19:36:00Z               1.064629          1.375167
13        frontendservice           trace.to_checkoutservice.duration_p95          25       0.774182              25              2  2024-01-19T19:36:00Z               1.064629          1.375167
14        checkoutservice          trace.to_currencyservice.duration_mean          25       0.003364              25              2  2024-01-19T19:28:00Z               0.004473          1.329889
15        currencyservice        trace.from_checkoutservice.duration_mean          25       0.003364              25              2  2024-01-19T19:28:00Z               0.004473          1.329889
16           emailservice                    trace.from_root.duration_p95          25       0.000087              25              2  2024-01-19T19:25:00Z               0.000109          1.254606
17                   root              trace.to_emailservice.duration_p95          25       0.000087              25              2  2024-01-19T19:25:00Z               0.000109          1.254606
18        frontendservice                    trace.from_root.duration_p95          25       0.857258              25              2  2024-01-19T19:34:00Z               1.064331          1.241553
19                   root           trace.to_frontendservice.duration_p95          25       0.857258              25              2  2024-01-19T19:34:00Z               1.064331          1.241553```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing variables if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure timestamps are parsed to UTC datetimes
for df in (log_df, error_df):
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    else:
        df['timestamp'] = df['timestamp'].dt.tz_convert('UTC')

# Parameters
services = ['currencyservice', 'checkoutservice', 'frontendservice', 'paymentservice', 'recommendationservice']
start_ts = pd.to_datetime('2024-01-19T19:12:14Z', utc=True)
end_ts   = pd.to_datetime('2024-01-19T19:42:14Z', utc=True)

# 1) Compute global P95 and full_count per (cmdb_id, log_name) using the entire series (no pre-filter)
log_full_counts = log_df.groupby(['cmdb_id', 'log_name'], dropna=False).size().reset_index(name='full_count')
log_p95 = log_df.groupby(['cmdb_id', 'log_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95_threshold')
log_full_stats_df = log_full_counts.merge(log_p95, on=['cmdb_id', 'log_name'])

# 2) Filter log rows to incident window and restrict to candidate services
incident_log_df = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts) & (log_df['cmdb_id'].isin(services))].copy()

# 3) Aggregate incident counts and max value
incident_agg_log = incident_log_df.groupby(['cmdb_id', 'log_name'], dropna=False).agg(
    incident_count=('value', 'size'),
    max_value_in_incident=('value', 'max')
).reset_index()

# Merge p95 thresholds into incident rows to flag anomalies
incident_with_p95_log = incident_log_df.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')

# Flag anomalies where value > p95_threshold (only when p95 exists)
incident_with_p95_log['is_anomaly'] = False
mask_valid_p95 = incident_with_p95_log['p95_threshold'].notna()
incident_with_p95_log.loc[mask_valid_p95, 'is_anomaly'] = incident_with_p95_log.loc[mask_valid_p95, 'value'] > incident_with_p95_log.loc[mask_valid_p95, 'p95_threshold']

# Compute anomaly counts and earliest anomaly timestamp per pair
anomaly_counts_log = incident_with_p95_log[incident_with_p95_log['is_anomaly']].groupby(
    ['cmdb_id', 'log_name'], dropna=False
).agg(
    anomaly_count=('is_anomaly', 'sum'),
    earliest_anomaly_ts=('timestamp', 'min')
).reset_index()

# Merge all aggregates
pair_incident_agg_log = log_full_stats_df.merge(incident_agg_log, on=['cmdb_id', 'log_name'], how='left')
pair_incident_agg_log = pair_incident_agg_log.merge(anomaly_counts_log, on=['cmdb_id', 'log_name'], how='left')

# Fill NaNs for pairs with no incident rows
pair_incident_agg_log['incident_count'] = pair_incident_agg_log['incident_count'].fillna(0).astype(int)
pair_incident_agg_log['max_value_in_incident'] = pair_incident_agg_log['max_value_in_incident']  # may be NaN
pair_incident_agg_log['anomaly_count'] = pair_incident_agg_log['anomaly_count'].fillna(0).astype(int)
pair_incident_agg_log['earliest_anomaly_ts'] = pair_incident_agg_log['earliest_anomaly_ts']  # may be NaT

# Compute ratio_max_to_p95 (None if p95 == 0 or p95 is NaN or max is NaN)
def compute_ratio_log(row):
    p95v = row['p95_threshold']
    maxv = row['max_value_in_incident']
    if pd.isna(p95v) or p95v == 0 or pd.isna(maxv):
        return None
    return float(maxv) / float(p95v)

pair_incident_agg_log['ratio_max_to_p95'] = pair_incident_agg_log.apply(compute_ratio_log, axis=1)

# Format earliest_anomaly_ts as ISO or None
pair_incident_agg_log['earliest_anomaly_ts_iso'] = pair_incident_agg_log['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
pair_incident_agg_log.loc[pair_incident_agg_log['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

# Keep only records with incident_count > 0 and restrict to candidate services
results_df_log = pair_incident_agg_log[(pair_incident_agg_log['incident_count'] > 0) & (pair_incident_agg_log['cmdb_id'].isin(services))].copy()

# Select requested columns and rename earliest column
results_df_log = results_df_log[[
    'cmdb_id', 'log_name', 'full_count', 'p95_threshold',
    'incident_count', 'anomaly_count', 'earliest_anomaly_ts_iso',
    'max_value_in_incident', 'ratio_max_to_p95'
]].rename(columns={'earliest_anomaly_ts_iso':'earliest_anomaly_ts'})

# Sort by anomaly_count desc then ratio desc, limit top 20
results_df_log = results_df_log.sort_values(by=['anomaly_count', 'ratio_max_to_p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# --- error_logs summary ---
# Overall total rows
error_total_rows = int(error_df.shape[0])

# Rows in incident window
error_incident_df = error_df[(error_df['timestamp'] >= start_ts) & (error_df['timestamp'] <= end_ts)].copy()
error_incident_rows = int(error_incident_df.shape[0])

# Top up to 10 (cmdb_id, message) counts (use groupby and sort)
if error_total_rows > 0:
    error_top_pairs_df = error_df.groupby(['cmdb_id', 'message'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(10).reset_index(drop=True)
else:
    error_top_pairs_df = pd.DataFrame(columns=['cmdb_id', 'message', 'count'])

# Compact summary DataFrame for error logs
error_summary_df = pd.DataFrame([{
    'error_total_rows': error_total_rows,
    'error_incident_rows': error_incident_rows
}])

# Keep intermediate DataFrames in kernel variables for follow-up:
# log_full_stats_df, incident_log_df, incident_with_p95_log, pair_incident_agg_log, results_df_log
# error_df, error_incident_df, error_top_pairs_df, error_summary_df

results_df_log, error_summary_df, error_top_pairs_df

```
Out[4]:
```
Summary (plain English)

- What I computed: global P95 per (cmdb_id, log_name) from the full log series, then inspected log rows for the incident window 2024-01-19T19:12:14Z → 2024-01-19T19:42:14Z for the services: currencyservice, checkoutservice, frontendservice, paymentservice, recommendationservice.

- Key findings (top results):
  - checkoutservice / log.row_count — full p95 = 110.4, incident_count = 25, anomaly_count = 2, earliest anomaly 2024-01-19T19:22:00Z, max_in_window = 117.0, ratio ≈ 1.0598 (≈6% above p95).
  - paymentservice / log.row_count — full p95 = 73.6, incident_count = 25, anomaly_count = 2, earliest anomaly 2024-01-19T19:22:00Z, max_in_window = 78.0, ratio ≈ 1.0598 (≈6% above p95).
  - recommendationservice / log.row_count — full p95 = 584.0, incident_count = 25, anomaly_count = 2, earliest anomaly 2024-01-19T19:17:00Z, max_in_window = 615.0, ratio ≈ 1.0531 (≈5.3% above p95).
  - currencyservice / log.row_count — full p95 = 2452.8, incident_count = 25, anomaly_count = 2, earliest anomaly 2024-01-19T19:17:00Z, max_in_window = 2480.0, ratio ≈ 1.0111 (≈1.1% above p95).

- log.error_count: for these services the global p95 is 0 and there are no anomalies in the window (incident_count = 25 but anomaly_count = 0 because values do not exceed p95).

- error_logs.csv: empty — total rows = 0, rows in incident window = 0, no (cmdb_id, message) entries.

- Interpretation: small, short-lived increases in log.row_count were observed across multiple services (2 samples above p95 per pair). The largest relative increase is ~6% for checkoutservice and paymentservice row_count; currencyservice shows only a ~1% bump. There are no error log spikes recorded.

- Notes: full intermediate DataFrames from the analysis are stored in the kernel for follow-up if you want deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  full_count  p95_threshold  incident_count  anomaly_count   earliest_anomaly_ts  max_value_in_incident  ratio_max_to_p95
0        checkoutservice    log.row_count          25          110.4              25              2  2024-01-19T19:22:00Z                  117.0          1.059783
1         paymentservice    log.row_count          25           73.6              25              2  2024-01-19T19:22:00Z                   78.0          1.059783
2  recommendationservice    log.row_count          25          584.0              25              2  2024-01-19T19:17:00Z                  615.0          1.053082
3        currencyservice    log.row_count          25         2452.8              25              2  2024-01-19T19:17:00Z                 2480.0          1.011089
4        checkoutservice  log.error_count          25            0.0              25              0                  None                    0.0               NaN
5        currencyservice  log.error_count          25            0.0              25              0                  None                    0.0               NaN
6         paymentservice  log.error_count          25            0.0              25              0                  None                    0.0               NaN
7  recommendationservice  log.error_count          25            0.0              25              0                  None                    0.0               NaN,    error_total_rows  error_incident_rows
0                 0                    0, Empty DataFrame
Columns: [cmdb_id, message, count]
Index: [])```
```